In [215]:
import polars as pl
import polars.selectors as cs
from typing import Any, Tuple
import time
import jax.numpy as jnp
import jax
import jax_dataloader as jdl
from flax import nnx as nnx
import optax
import wandb
from flax.training import train_state
import orbax.checkpoint as ocp
from sklearn.model_selection import KFold

In [207]:
cat_columns = {
    "MSSubClass": pl.Categorical,
    "MSZoning": pl.Categorical,
    "Street": pl.Categorical,
    "Alley": pl.Categorical,
    "LotShape": pl.Categorical,
    "LandContour": pl.Categorical,
    "Utilities": pl.Categorical,
    "LotConfig": pl.Categorical,
    "LandSlope": pl.Categorical,
    "Neighborhood": pl.Categorical,
    "Condition1": pl.Categorical,
    "Condition2": pl.Categorical,
    "BldgType": pl.Categorical,
    "HouseStyle": pl.Categorical,
    "OverallQual": pl.Categorical,
    "OverallCond": pl.Categorical,
    "RoofStyle": pl.Categorical,
    "RoofMatl": pl.Categorical,
    "Exterior1st": pl.Categorical,
    "Exterior2nd": pl.Categorical,
    "MasVnrType": pl.Categorical,
    "ExterQual": pl.Categorical,
    "ExterCond": pl.Categorical,
    "Foundation": pl.Categorical,
    "BsmtQual": pl.Categorical,
    "BsmtCond": pl.Categorical,
    "BsmtExposure": pl.Categorical,
    "BsmtFinType1": pl.Categorical,
    "BsmtFinType2": pl.Categorical,
    "Heating": pl.Categorical,
    "HeatingQC": pl.Categorical,
    "CentralAir": pl.Categorical,
    "Electrical": pl.Categorical,
    "KitchenQual": pl.Categorical,
    "Functional": pl.Categorical,
    "FireplaceQu": pl.Categorical,
    "GarageType": pl.Categorical,
    "GarageFinish": pl.Categorical,
    "GarageQual": pl.Categorical,
    "GarageCond": pl.Categorical,
    "PavedDrive": pl.Categorical,
    "PoolQC": pl.Categorical,
    "Fence": pl.Categorical,
    "MiscFeature": pl.Categorical,
    "SaleType": pl.Categorical,
    "SaleCondition": pl.Categorical,
}


In [276]:
raw_train_df = pl.read_csv("./train.csv",
                     null_values="NA",
                     schema_overrides=cat_columns)

In [278]:
def process_df(df: pl.DataFrame):
    df = df.drop("Id")

    df = df.with_columns(
        (pl.col("GarageYrBlt").is_null().alias("HasGarage")),
        (pl.col("YrSold") - pl.col("GarageYrBlt"))
        .alias("GarageYrBlt"),
        (pl.col("YrSold") - pl.col("YearBuilt"))
        .alias("Age"),
        (pl.col("TotalBsmtSF") + pl.col("GrLivArea"))
        .alias("TotalSF"),
        pl.col("SalePrice").log()
    )

    df = df.with_columns(
        ((cs.numeric() - cs.numeric().mean()) / cs.numeric().std()).exclude("SalePrice"),
    )

    df = df.fill_null(0) # We fill LotFrontage and MasVnrArea NA with 0

    df = df.to_dummies(columns=cs.categorical())

    df = df.select(
        pl.all().exclude("SalePrice"),
        pl.col("SalePrice")
    )

    return df

train_df = process_df(raw_train_df)

In [279]:
def loss_fn(model, batch):
    x, y = batch
    y_pred = model(x).squeeze(-1)
    # jax.debug.print("y {}", y)
    # print(y_pred.shape, y.shape)
    assert y_pred.shape == y.shape
    return jnp.sqrt(jnp.mean((y_pred - y) ** 2))

@nnx.jit
def train_step(model: nnx.Module, optimiser: nnx.Optimizer, metrics: nnx.MultiMetric, batch: Tuple[jax.Array, jax.Array]):
    model.train()
    loss, grads = nnx.value_and_grad(loss_fn)(model, batch)
    metrics.update(loss=loss)
    optimiser.update(grads)

@nnx.jit
def eval_step(model: nnx.Module, metrics: nnx.MultiMetric, batch: tuple[jax.Array, jax.Array]):
    model.eval()
    loss = loss_fn(model, batch)
    metrics.update(loss=loss)

In [ ]:
# Using linear net for now

# class MLP(nnx.Module):
#     def __init__(self, din, dhidden, dout, rngs: nnx.Rngs):
#         self.linear1 = nnx.Linear(din, dhidden, rngs=rngs)
#         self.drop1 = nnx.Dropout(0.3, rngs=rngs)
#         self.linear2 = nnx.Linear(dhidden, dhidden, rngs=rngs)
#         self.drop2 = nnx.Dropout(0.5, rngs=rngs)
#         self.linear3 = nnx.Linear(dhidden, dout, rngs=rngs)

#     def __call__(self, X):
#         H1 = nnx.relu(self.linear1(X))
#         H1 = self.drop1(H1)
#         H2 = nnx.relu(self.linear2(H1))
#         H2 = self.drop2(H2)
#         return self.linear3(H2)


In [282]:
def train_model(epochs, dl_train, dl_val):
    rngs = nnx.Rngs(default=0)
    # model = MLP(336, 512, 1, rngs=rngs)
    model = nnx.Linear(338, 1, rngs=rngs)
    optimiser = nnx.Optimizer(model, optax.sgd(1e-2))
    metrics = nnx.MultiMetric(
        loss = nnx.metrics.Average("loss")
    )

    final_val_loss = 0

    for i in range(epochs):
        for batch in dl_train:
            train_step(model, optimiser, metrics, batch)

        m = metrics.compute()
        metrics.reset()
        # print(f"epoch {i+1} of {epochs} with loss = {m["loss"]}")

        for batch in dl_val:
            eval_step(model, metrics, batch)

        m = metrics.compute()
        metrics.reset()
        # print(f"epoch {i+1} of {epochs} with loss = {m["loss"]}")
        final_val_loss = m["loss"].item()

    return final_val_loss
        

In [289]:
kf = KFold(n_splits=5, shuffle=True)

arr = train_df.to_jax()
losses = jnp.array([])

for i, (train_index, test_index) in enumerate(kf.split(arr)): # type: ignore
    X_train, y_train = arr[train_index, :-1], arr[train_index, -1]
    X_val,   y_val   = arr[test_index,  :-1], arr[test_index,  -1]

    train_ds = jdl.ArrayDataset(X_train, y_train)
    val_ds   = jdl.ArrayDataset(X_val,   y_val)

    dl_train = jdl.DataLoader(train_ds, 'jax',
                              batch_size=64, shuffle=True)
    dl_val   = jdl.DataLoader(val_ds,   'jax',
                              batch_size=64, shuffle=False)
    
    loss = train_model(100, dl_train, dl_val)
    losses = jnp.append(losses, loss)

    print(f"fold {i} loss = {loss}")

print(f"final loss = {losses.mean()}")

fold 0 loss = 0.20870980620384216
fold 1 loss = 0.21215160191059113
fold 2 loss = 0.19739001989364624
fold 3 loss = 0.1819535493850708
fold 4 loss = 0.22027790546417236
final loss = 0.20409658551216125
